<a href="https://colab.research.google.com/github/MarkusThill/techdays26/blob/main/torch_ntuple_net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TODO:
- Build plots for BitBullyArena Results!

In [ ]:
!ssh-keygen -t rsa -b 4096 -f ~/.ssh/id_rsa -N "" -q
!ssh-keyscan -t rsa github.com >> ~/.ssh/known_hosts
!cat ~/.ssh/id_rsa.pub

Add key here:
https://github.com/settings/keys

In [ ]:
!ssh -T git@github.com
!git config --global user.email "8896197+MarkusThill@users.noreply.github.com"
!git config --global user.name "Markus Thill"
!git clone git@github.com:MarkusThill/techdays26.git
!pip install -e techdays26[dev,lab1]

In [ ]:
# Only relevant for Google Colab:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import output

    output.enable_custom_widget_manager()

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import ClassVar
from __future__ import annotations

from dataclasses import dataclass
from typing import ClassVar

import torch


def _i64(x: int) -> int:
    return x


@dataclass(slots=True)
class BoardBatch:
    all_tokens: torch.Tensor  # [B] int64
    active_tokens: torch.Tensor  # [B] int64
    moves_left: torch.Tensor  # [B] int16/int32

    N_COLUMNS: ClassVar[int] = 7
    N_ROWS: ClassVar[int] = 6
    COLUMN_BIT_OFFSET: ClassVar[int] = 9

    BB_BOTTOM_ROW: ClassVar[int] = _i64(
        (1 << 54) | (1 << 45) | (1 << 36) | (1 << 27) | (1 << 18) | (1 << 9) | (1 << 0)
    )
    BB_ALL_LEGAL_TOKENS: ClassVar[int] = _i64(
        sum(1 << (c * 9 + r) for c in range(7) for r in range(6))
    )

    # -------- caching (per-process) --------
    _WEIGHTS_CACHE: ClassVar[dict[tuple[str, int | None, int], torch.Tensor]] = {}
    _PATTERN_MASKS_CACHE: ClassVar[dict[tuple[str, int | None, int], torch.Tensor]] = {}
    _COL_MASKS_CACHE: ClassVar[dict[tuple[str, int | None], torch.Tensor]] = {}

    @staticmethod
    def _dev_key(dev: torch.device) -> tuple[str, int | None]:
        # e.g. ("cpu", None) or ("cuda", 0)
        return (dev.type, dev.index)

    @classmethod
    def _col_masks(cls, dev: torch.device) -> torch.Tensor:
        """[7] int64 masks for each column's 6 playable bits."""
        dkey = cls._dev_key(dev)
        if dkey in cls._COL_MASKS_CACHE:
            return cls._COL_MASKS_CACHE[dkey]

        row_mask = (1 << cls.N_ROWS) - 1  # 0b111111
        cols = torch.arange(cls.N_COLUMNS, device=dev, dtype=torch.int64)
        base = cols * cls.COLUMN_BIT_OFFSET
        col_masks = torch.tensor(row_mask, device=dev, dtype=torch.int64) << base  # [7]
        cls._COL_MASKS_CACHE[dkey] = col_masks
        return col_masks

    @classmethod
    def _all_legal_mask(cls, dev: torch.device) -> torch.Tensor:
        """Scalar int64 mask with all legal playable squares set."""
        col_masks = cls._col_masks(dev)
        return col_masks.sum().to(dtype=torch.int64)

    @classmethod
    def _weights_base4(cls, dev: torch.device, n: int) -> torch.Tensor:
        """[1,1,n] int64 weights = 4**i cached per device and n."""
        dkey = cls._dev_key(dev)
        key = (dkey[0], dkey[1], n)
        w = cls._WEIGHTS_CACHE.get(key)
        if w is None:
            w = (4 ** torch.arange(n, device=dev, dtype=torch.int64)).view(1, 1, n)
            cls._WEIGHTS_CACHE[key] = w
        return w

    @classmethod
    def _pattern_masks(
        cls, patterns_bitidx: torch.Tensor, dev: torch.device
    ) -> torch.Tensor:
        """[1,M,N] int64 one-hot bit masks for patterns, cached per device + pattern object id."""
        if patterns_bitidx.dtype != torch.int64:
            raise TypeError("patterns_bitidx must be int64 (bit indices 0..63).")
        if patterns_bitidx.device != dev:
            # you can decide: enforce caller moves it, or auto-move.
            patterns_bitidx = patterns_bitidx.to(device=dev)

        dkey = cls._dev_key(dev)
        key = (dkey[0], dkey[1], id(patterns_bitidx))

        masks = cls._PATTERN_MASKS_CACHE.get(key)
        if masks is None:
            pat = patterns_bitidx.view(1, *patterns_bitidx.shape)  # [1,M,N]
            one = torch.ones((), device=dev, dtype=torch.int64)
            masks = one << pat  # [1,M,N] int64
            cls._PATTERN_MASKS_CACHE[key] = masks
        return masks

    def legal_moves_mask(self) -> torch.Tensor:
        """[B] int64 landing squares (reachable in next move)."""
        dev = self.all_tokens.device
        bottom = torch.full((), self.BB_BOTTOM_ROW, device=dev, dtype=torch.int64)
        all_legal = self._all_legal_mask(dev)
        return (self.all_tokens + bottom) & all_legal

    def _legal_moves_mask(self) -> torch.Tensor:
        return self.legal_moves_mask()
        # dev = self._device_check(self.all_tokens, self.active_tokens, self.moves_left)
        # bottom = torch.full((), self.BB_BOTTOM_ROW, device=dev, dtype=torch.int64)
        # legal_sq = torch.full((), self.BB_ALL_LEGAL_TOKENS, device=dev, dtype=torch.int64)
        # return (self.all_tokens + bottom) & legal_sq  # [B] landing squares in each column

    def table_positions(self, patterns_bitidx: torch.Tensor) -> torch.Tensor:
        """Compute T for each board and each pattern (n-tuple).

        patterns_bitidx:
            [M,N] int64 tensor of bit indices (0..63) in your bitboard layout.

        Returns:
            T: [B,M] int64
        """
        dev = self.all_tokens.device
        B = self.all_tokens.shape[0]

        # cached: [1,M,N] one-hot bit masks
        masks = self._pattern_masks(patterns_bitidx, dev)  # [1,M,N]
        M, N = patterns_bitidx.shape

        all_tok = self.all_tokens.view(B, 1, 1)
        act_tok = self.active_tokens.view(B, 1, 1)

        occupied = (all_tok & masks) != 0  # [B,M,N] bool
        is_active = (act_tok & masks) != 0  # [B,M,N] bool

        # reachable = landing squares only
        reachable_mask = self.legal_moves_mask().view(B, 1, 1)  # [B,1,1]
        reachable = (~occupied) & ((reachable_mask & masks) != 0)  # [B,M,N] bool

        # Determine which color is "active player" from moves_left parity (matches your C++)
        active_is_yellow = (self.moves_left.to(torch.int64) & 1) == 0  # [B] bool

        # If active is yellow: active tokens are yellow, else active tokens are red.
        yellow = (
            torch.where(active_is_yellow.view(B, 1, 1), is_active, ~is_active)
            & occupied
        )
        red = occupied & ~yellow

        # s in {0,1,2,3}
        s = torch.zeros((B, M, N), device=dev, dtype=torch.int64)
        s = torch.where(reachable, torch.full_like(s, 3), s)
        s = torch.where(yellow, torch.full_like(s, 1), s)
        s = torch.where(red, torch.full_like(s, 2), s)

        # cached weights: [1,1,N]
        w = self._weights_base4(dev, N)
        return (s * w).sum(dim=2)  # [B,M]

    @staticmethod
    def _device_check(*tensors: torch.Tensor) -> torch.device:
        dev = tensors[0].device
        for t in tensors[1:]:
            if t.device != dev:
                raise ValueError("All tensors must be on the same device.")
        return dev

    @classmethod
    def empty(
        cls,
        batch_size: int,
        device: torch.device | str,
        *,
        moves_left_dtype: torch.dtype = torch.int16,
    ) -> BoardBatch:
        dev = torch.device(device)
        all_tokens = torch.zeros(batch_size, device=dev, dtype=torch.int64)
        active_tokens = torch.zeros(batch_size, device=dev, dtype=torch.int64)
        moves_left = torch.full(
            (batch_size,),
            cls.N_COLUMNS * cls.N_ROWS,
            device=dev,
            dtype=moves_left_dtype,
        )
        return cls(
            all_tokens=all_tokens, active_tokens=active_tokens, moves_left=moves_left
        )

    @staticmethod
    def _pow2(shift: torch.Tensor) -> torch.Tensor:
        # shift int64 in [0, 62] -> safe in int64
        return torch.ones_like(shift, dtype=torch.int64) << shift

    def _apply_move(self, mv: torch.Tensor, legal: torch.Tensor) -> torch.Tensor:
        """Apply move bitboards (mv) where legal, in-place. Returns legal mask."""
        dev = self._device_check(
            self.all_tokens, self.active_tokens, self.moves_left, mv, legal
        )
        mv = mv.to(device=dev, dtype=torch.int64)
        legal = legal.to(device=dev, dtype=torch.bool)

        mask = -legal.to(torch.int64)  # -1 where legal else 0
        mv = mv & mask  # illegal -> 0

        # strict C++ semantics: only switch player if legal
        self.active_tokens ^= self.all_tokens & mask
        self.all_tokens ^= mv
        self.moves_left -= legal.to(self.moves_left.dtype)
        return legal

    # ---------------------------------------------------------------------
    # Play by columns (0..6), returns [B] bool legal
    # ---------------------------------------------------------------------
    def play_columns(self, columns: torch.Tensor) -> torch.Tensor:
        dev = self._device_check(
            self.all_tokens, self.active_tokens, self.moves_left, columns
        )
        cols = columns.to(device=dev, dtype=torch.int64)

        in_range = (cols >= 0) & (cols < self.N_COLUMNS)
        cols_c = cols.clamp(0, self.N_COLUMNS - 1)

        # top cell in that column must be empty
        top_shift = cols_c * self.COLUMN_BIT_OFFSET + (self.N_ROWS - 1)
        top = self._pow2(top_shift)
        not_full = (self.all_tokens & top) == 0
        legal = in_range & not_full

        # column mask: (1<<(base+N_ROWS)) - (1<<base)
        base = cols_c * self.COLUMN_BIT_OFFSET
        lo = self._pow2(base)
        hi = self._pow2(base + self.N_ROWS)
        col_mask = hi - lo

        bottom = torch.full((), self.BB_BOTTOM_ROW, device=dev, dtype=torch.int64)

        # landing square
        mv = (self.all_tokens + bottom) & col_mask
        return self._apply_move(mv, legal)

    # ---------------------------------------------------------------------
    # Play by move masks (one-hot landing square per board), returns [B] bool legal
    # ---------------------------------------------------------------------
    def play_masks(self, mv: torch.Tensor) -> torch.Tensor:
        """mv: [B] int64, expected one bit set at the landing square."""
        dev = self._device_check(
            self.all_tokens, self.active_tokens, self.moves_left, mv
        )
        mv = mv.to(device=dev, dtype=torch.int64)

        # Legal if:
        # 1) mv is non-zero
        # 2) mv is subset of legal landing squares (exactly those bits from legalMovesMask)
        legal_moves = self._legal_moves_mask()  # [B]
        legal = (mv != 0) & ((mv & legal_moves) == mv)  # mv ⊆ legal_moves

        # Optionally enforce "single bit" (one-hot). This costs popcount; skip for speed.
        # If you want a cheap-ish check: mv & (mv-1) == 0 (works for signed too if mv>0).
        # one_hot = (mv > 0) & ((mv & (mv - 1)) == 0)
        # legal &= one_hot

        return self._apply_move(mv, legal)

    # Backwards-compatible name
    def play(self, columns: torch.Tensor) -> torch.Tensor:
        return self.play_columns(columns)

    def has_win(self) -> torch.Tensor:
        y = self.active_tokens ^ self.all_tokens  # player who just moved

        x = y & (y << 2)
        win = (x & (x << 1)) != 0

        off = self.COLUMN_BIT_OFFSET

        x = y & (y << (2 * off))
        win |= (x & (x << off)) != 0

        d1 = off - 1
        x = y & (y << (2 * d1))
        win |= (x & (x << d1)) != 0

        d2 = off + 1
        x = y & (y << (2 * d2))
        win |= (x & (x << d2)) != 0

        return win

    def winning_positions(self, x: torch.Tensor, verticals: bool) -> torch.Tensor:
        """[B] int64 bitboard of winning squares for player bitboard x."""
        dev = self._device_check(
            self.all_tokens, self.active_tokens, self.moves_left, x
        )
        x = x.to(device=dev, dtype=torch.int64)

        all_legal = torch.full(
            (), self.BB_ALL_LEGAL_TOKENS, device=dev, dtype=torch.int64
        )

        wins = torch.zeros_like(x)

        if verticals:
            wins |= (x << 1) & (x << 2) & (x << 3)

        off = self.COLUMN_BIT_OFFSET
        for b in (off - 1, off, off + 1):
            # left-ish directions
            tmp = (x << b) & (x << (2 * b))
            wins |= tmp & (x << (3 * b))
            wins |= tmp & (x >> b)

            # right-ish directions
            tmp = (x >> b) & (x >> (2 * b))
            wins |= tmp & (x << b)
            wins |= tmp & (x >> (3 * b))

        return wins & all_legal

    def can_win(self) -> torch.Tensor:
        """[B] bool: whether side to move has any immediate winning move."""
        dev = self._device_check(self.all_tokens, self.active_tokens, self.moves_left)
        wins = self.winning_positions(self.active_tokens, verticals=True)  # [B] int64
        moves = self._legal_moves_mask()  # [B] int64
        return (wins & moves) != 0

    def generate_legal_moves(self) -> torch.Tensor:
        """[B] int64: all legal landing squares (one bit per legal column)."""
        return self._legal_moves_mask()

    def generate_non_losing_moves(self) -> torch.Tensor:
        """[B] int64: non-losing move bitboard (may be 0 for forced-loss positions)."""
        dev = self._device_check(self.all_tokens, self.active_tokens, self.moves_left)
        moves = self._legal_moves_mask()  # landing squares for all columns

        # Opponent stones = active ^ all  (same as C++)
        opp = self.active_tokens ^ self.all_tokens

        threats = self.winning_positions(opp, verticals=True)

        direct_threats = threats & moves  # moves that block immediate opponent win
        has_direct = direct_threats != 0

        # "more than one direct threat?"  => (x & (x-1)) != 0  (same as C++)
        multi = (direct_threats & (direct_threats - 1)) != 0

        # If there is a direct threat:
        #   - if multiple: moves = 0
        #   - else:       moves = direct_threats
        moves = torch.where(
            has_direct,
            torch.where(multi, torch.zeros_like(moves), direct_threats),
            moves,
        )

        # No token under an opponent's threat.
        return moves & ~(threats >> 1)

    def can_win_column(self, columns: torch.Tensor) -> torch.Tensor:
        """[B] bool: whether playing `columns[b]` wins immediately for each board."""
        dev = self._device_check(
            self.all_tokens, self.active_tokens, self.moves_left, columns
        )
        cols = columns.to(device=dev, dtype=torch.int64)

        in_range = (cols >= 0) & (cols < self.N_COLUMNS)
        cols_c = cols.clamp(0, self.N_COLUMNS - 1)

        # isLegalMove(column): check top cell in that column is empty
        top_shift = cols_c * self.COLUMN_BIT_OFFSET + (self.N_ROWS - 1)
        top = self._pow2(top_shift)  # [B] int64
        not_full = (self.all_tokens & top) == 0
        legal = in_range & not_full

        # getColumnMask(column): (1<<(base+N_ROWS)) - (1<<base)
        base = cols_c * self.COLUMN_BIT_OFFSET
        lo = self._pow2(base)
        hi = self._pow2(base + self.N_ROWS)
        col_mask = hi - lo

        wins = self.winning_positions(self.active_tokens, verticals=True)  # [B] int64
        moves = self._legal_moves_mask()  # [B] int64

        return legal & ((wins & moves & col_mask) != 0)

    def reset(self, done: torch.Tensor) -> None:
        """Reset boards where `done` is True back to the empty position (in-place).

        Args:
            done: [B] bool tensor. True means "reset this board".

        Notes:
            - No CPU sync.
            - No advanced indexing.
            - Keeps tensors allocated; only writes values.
        """
        dev = self._device_check(
            self.all_tokens, self.active_tokens, self.moves_left, done
        )
        d = done.to(device=dev, dtype=torch.bool)

        # mask: -1 where done else 0  (int64)
        m = -d.to(torch.int64)

        # Clear bitboards for done boards; keep others unchanged.
        # keep_mask is 0 for done, -1 for keep (all bits set).
        keep_mask = ~m
        self.all_tokens &= keep_mask
        self.active_tokens &= keep_mask

        # Reset moves_left for done boards to 42 (full empty board).
        full = torch.full(
            (), self.N_COLUMNS * self.N_ROWS, device=dev, dtype=self.moves_left.dtype
        )
        self.moves_left = torch.where(d, full, self.moves_left)

    def iter_move_masks(self, moves: torch.Tensor | None = None, *, max_moves: int = 7):
        """Yield one-hot move masks from a move-set bitboard.

        Args:
            moves:
                [B] int64 tensor where each entry is a bitboard with <= 7 bits set.
                If None, uses `self.generate_non_losing_moves()`.
            max_moves:
                Upper bound on number of yielded moves (Connect-4: 7).

        Yields:
            mv:
                [B] int64 tensor where each entry is either 0 (no move for that board
                in this iteration) or a one-hot bitboard with a single bit set.

        Example:
            Iterate non-losing moves and apply them on copies:
            ```python
            nl = board.generate_non_losing_moves()
            for mv in board.iter_move_masks(nl):
                active = mv != 0
                if not active.any():
                    break

                tmp = BoardBatch(
                    all_tokens=board.all_tokens.clone(),
                    active_tokens=board.active_tokens.clone(),
                    moves_left=board.moves_left.clone(),
                )
                tmp.play_masks(mv)
                win_after = tmp.has_win()
            ```
        """
        if moves is None:
            moves = self.generate_non_losing_moves()

        dev = self._device_check(
            self.all_tokens, self.active_tokens, self.moves_left, moves
        )
        m = moves.to(device=dev, dtype=torch.int64)

        for _ in range(max_moves):
            mv = m & -m  # extract lsb (one-hot)
            yield mv
            m = m ^ mv  # clear bit

    def mirror(self) -> "BoardBatch":
        """Return a horizontally mirrored copy of the batch (0<->6, 1<->5, 2<->4)."""

        def mirror_bits(x: torch.Tensor) -> torch.Tensor:
            # Column masks for the 6 playable bits in each column.
            # mask(c) = ((1<<N_ROWS)-1) << (c*COLUMN_BIT_OFFSET)
            row_mask = (1 << self.N_ROWS) - 1  # 0b111111
            m0 = row_mask << (0 * self.COLUMN_BIT_OFFSET)
            m1 = row_mask << (1 * self.COLUMN_BIT_OFFSET)
            m2 = row_mask << (2 * self.COLUMN_BIT_OFFSET)
            m3 = row_mask << (3 * self.COLUMN_BIT_OFFSET)
            m4 = row_mask << (4 * self.COLUMN_BIT_OFFSET)
            m5 = row_mask << (5 * self.COLUMN_BIT_OFFSET)
            m6 = row_mask << (6 * self.COLUMN_BIT_OFFSET)

            s54 = 6 * self.COLUMN_BIT_OFFSET  # 54
            s36 = 4 * self.COLUMN_BIT_OFFSET  # 36
            s18 = 2 * self.COLUMN_BIT_OFFSET  # 18

            # Same logic as the C++ mirrorBitBoard():
            y = torch.zeros_like(x)
            y |= (x & m6) >> s54
            y |= (x & m0) << s54

            y |= (x & m5) >> s36
            y |= (x & m1) << s36

            y |= (x & m4) >> s18
            y |= (x & m2) << s18

            y |= x & m3  # center column unchanged
            return y

        return BoardBatch(
            all_tokens=mirror_bits(self.all_tokens),
            active_tokens=mirror_bits(self.active_tokens),
            moves_left=self.moves_left.clone(),  # preserve dtype; keep separate tensor
        )

    def reward(self) -> torch.Tensor:
        """
        Returns:
            [B] float tensor with:
                +1.0  -> yellow wins
                -1.0  -> red wins
                0.0  -> draw
                NaN  -> game not finished
        """
        dev = self._device_check(self.all_tokens, self.active_tokens, self.moves_left)

        B = self.moves_left.shape[0]
        r = torch.full((B,), float("nan"), device=dev)

        win = self.has_win()  # last mover won
        draw = (self.moves_left == 0) & ~win

        # moves_left AFTER move:
        # odd -> yellow just moved (yellow starts and places the first token)
        # even  -> red just moved
        yellow_just_moved = (self.moves_left.to(torch.int64) & 1) == 1

        # Assign rewards
        r = torch.where(win & yellow_just_moved, torch.tensor(1.0, device=dev), r)
        r = torch.where(win & ~yellow_just_moved, torch.tensor(-1.0, device=dev), r)
        r = torch.where(draw, torch.tensor(0.0, device=dev), r)

        return r

    def active_player(self) -> torch.Tensor:
        """
        Returns:
            [B] int8 tensor:
                1 -> Yellow (starting player)
                2 -> Red (second player)
        """
        dev = self._device_check(self.all_tokens, self.active_tokens, self.moves_left)

        # Even moves_left -> Yellow to move
        yellow_to_move = (self.moves_left.to(torch.int64) & 1) == 0

        return torch.where(
            yellow_to_move,
            torch.ones_like(self.moves_left, dtype=torch.int8, device=dev),
            torch.full_like(self.moves_left, 2, dtype=torch.int8, device=dev),
        )

    def active_player_sign(self) -> torch.Tensor:
        """
        Returns:
            [B] float32 tensor:
                +1.0 -> Yellow to move
                -1.0 -> Red to move
        """
        dev = self._device_check(self.all_tokens, self.active_tokens, self.moves_left)

        # Even moves_left -> Yellow to move
        yellow_to_move = (self.moves_left.to(torch.int64) & 1) == 0

        return torch.where(
            yellow_to_move,
            torch.ones_like(self.moves_left, dtype=torch.float32, device=dev),
            -torch.ones_like(self.moves_left, dtype=torch.float32, device=dev),
        )

    def done(self) -> torch.Tensor:
        return self.has_win() | (self.moves_left <= 0)

In [ ]:
def move_mask_to_column(mv: int, *, column_bit_offset: int = 9) -> int:
    """Return the column index (0..6) for a one-hot move mask.

    Args:
        mv: int64 bitboard with exactly one bit set (landing square).
        column_bit_offset: Bit stride between columns (default: 9).

    Returns:
        int: Column index (0..6).

    Raises:
        ValueError: If mv == 0 or mv has more than one bit set.
    """
    if mv == 0 or (mv & (mv - 1)) != 0:
        raise ValueError(f"mv must be one-hot, got {mv:#x}")

    bit_index = mv.bit_length() - 1
    return bit_index // column_bit_offset

In [ ]:
import bitbully.bitbully_core as bbc

device = "cuda"
B = 100
reset_done_boards = True
compare_with_bitbully = True

torch_board = BoardBatch.empty(B, device)
bb_board = [bbc.BoardCore() for _ in range(B)]


for i in range(42 * 1):
    # if device == "cuda": torch.cuda.synchronize()
    actions = torch.randint(0, 7, (B,), device=device)
    legal = torch_board.play(actions)
    won = torch_board.has_win()
    done = torch_board.done()
    non_losing_moves = torch_board.generate_non_losing_moves()
    legal_moves = torch_board.generate_legal_moves()
    can_win = torch_board.can_win()

    can_win_actions = torch.randint(0, 7, (B,), device=device)
    can_win_column = torch_board.can_win_column(can_win_actions)

    # Mirror sanity check
    torch_board_mir = torch_board.mirror()

    # rewards
    reward = torch_board.reward()

    # active player
    active_player = torch_board.active_player()
    active_player_sign = torch_board.active_player_sign()

    # sanity: mirroring twice gives original (bitboards)
    b3 = torch_board_mir.mirror()
    assert torch.equal(b3.all_tokens, torch_board.all_tokens)
    assert torch.equal(b3.active_tokens, torch_board.active_tokens)
    assert torch.equal(b3.moves_left, torch_board.moves_left)

    # rewards
    reward = torch_board.reward()

    # if device == "cuda": torch.cuda.synchronize()

    # bitbully comparisons
    if compare_with_bitbully:
        for b_idx in range(B):
            a = actions[b_idx].item()
            ll = bb_board[b_idx].play(a)
            all_tokens, active_tokens, moves_left = bb_board[b_idx].rawState()
            assert (all_tokens, active_tokens, moves_left) == (
                torch_board.all_tokens[b_idx].item(),
                torch_board.active_tokens[b_idx].item(),
                torch_board.moves_left[b_idx].item(),
            ), f"Problem with {b_idx}"
            assert ll == legal[b_idx].item(), f"Problem with {b_idx}"
            assert bb_board[b_idx].hasWin() == won[b_idx].item(), (
                f"Problem with {b_idx}"
            )

            # rewards
            if reward[b_idx].item() != reward[b_idx].item():  # <- check for nan
                assert (
                    not bb_board[b_idx].hasWin() and bb_board[b_idx].movesLeft() > 0
                ), f"Problem with {b_idx}"
            elif reward[b_idx].item() == 1.0:
                assert (
                    bb_board[b_idx].hasWin() and bb_board[b_idx].movesLeft() % 2 == 1
                ), f"Problem with {b_idx}"
            elif reward[b_idx].item() == -1.0:
                assert (
                    bb_board[b_idx].hasWin() and bb_board[b_idx].movesLeft() % 2 == 0
                ), f"Problem with {b_idx}"
            elif reward[b_idx].item() == 0.0:
                assert (
                    not bb_board[b_idx].hasWin() and bb_board[b_idx].movesLeft() == 0
                ), f"Problem with {b_idx}"
            else:
                assert False, f"Problem with {b_idx}"

            # active player
            assert (
                bb_board[b_idx].popCountBoard() % 2 + 1 == active_player[b_idx].item()
            ), f"Problem with {b_idx}"
            assert (
                1 - 2 * (bb_board[b_idx].popCountBoard() % 2)
                == active_player_sign[b_idx].item()
            ), f"Problem with {b_idx}"

            # moves
            assert (
                bb_board[b_idx].generateNonLosingMoves() == non_losing_moves[b_idx]
            ), f"Problem with {b_idx}"
            assert bb_board[b_idx].legalMovesMask() == legal_moves[b_idx], (
                f"Problem with {b_idx}"
            )

            # can_win
            assert bb_board[b_idx].canWin() == can_win[b_idx].item(), (
                f"Problem with {b_idx}"
            )

            # can win column:
            assert (
                bb_board[b_idx].canWin(can_win_actions[b_idx].item())
                == can_win_column[b_idx].item()
            ), f"Problem with {b_idx}"

            # mirrored board:
            all_tokens_mir, active_tokens_mir, moves_left_mir = (
                bb_board[b_idx].mirror().rawState()
            )
            assert (all_tokens_mir, active_tokens_mir, moves_left_mir) == (
                torch_board_mir.all_tokens[b_idx].item(),
                torch_board_mir.active_tokens[b_idx].item(),
                torch_board_mir.moves_left[b_idx].item(),
            ), f"Problem with {b_idx}"

        if reset_done_boards:
            for b_idx in range(B):
                bb_done = bb_board[b_idx].hasWin() or bb_board[b_idx].movesLeft() <= 0
                assert bb_done == done[b_idx].item(), f"Problem with {b_idx}"
                if bb_done:
                    bb_board[b_idx].setRawState(0, 0, 42)

    if reset_done_boards:
        torch_board.reset(done)

    if compare_with_bitbully:
        # Check all after_states
        nl = torch_board.generate_non_losing_moves()
        for mv in torch_board.iter_move_masks(nl):
            active = mv != 0
            if not active.any():
                break

            tmp = BoardBatch(
                all_tokens=torch_board.all_tokens.clone(),
                active_tokens=torch_board.active_tokens.clone(),
                moves_left=torch_board.moves_left.clone(),
            )
            legal = tmp.play_masks(mv)
            won = tmp.has_win()
            done = tmp.done()
            non_losing_moves = tmp.generate_non_losing_moves()
            can_win = tmp.can_win()

            can_win_actions = torch.randint(0, 7, (B,), device=device)
            can_win_column = tmp.can_win_column(can_win_actions)

            for b_idx in range(B):
                mv_onehot = mv[b_idx].item()
                if (
                    mv_onehot
                ):  # TODO: BitBully-Core should also return masks (needs lib change)
                    a = move_mask_to_column(mv_onehot)
                    bb_new_board = bb_board[b_idx].playMoveOnCopy(a)
                    ll = bb_new_board.movesLeft() < 42
                else:
                    bb_new_board = bb_board[b_idx].copy()
                    ll = False

                all_tokens, active_tokens, moves_left = bb_new_board.rawState()
                assert (all_tokens, active_tokens, moves_left) == (
                    tmp.all_tokens[b_idx].item(),
                    tmp.active_tokens[b_idx].item(),
                    tmp.moves_left[b_idx].item(),
                ), f"Problem with {b_idx}"
                assert bb_new_board.hasWin() == won[b_idx].item(), (
                    f"Problem with {b_idx}"
                )
                assert ll == legal[b_idx].item(), f"Problem with {b_idx}"

                # Check equaility on winning_positions
                assert (
                    bb_new_board.generateNonLosingMoves() == non_losing_moves[b_idx]
                ), f"Problem with {b_idx}"

                # can_win
                assert bb_new_board.canWin() == can_win[b_idx].item(), (
                    f"Problem with {b_idx}"
                )

                # can win column:
                assert (
                    bb_new_board.canWin(can_win_actions[b_idx].item())
                    == can_win_column[b_idx].item()
                ), f"Problem with {b_idx}"

In [ ]:
# Standard Tuple-Set
# Uses this Board representation:
# 5 11 17 23 29 35 41
# 4 10 16 22 28 34 40
# 3  9 15 21 27 33 39
# 2  8 14 20 26 32 38
# 1  7 13 19 25 31 37
# 0  6 12 18 24 30 36

# Our Layout:
# [ *,  *,  *,  *,  *,  *,  *]
# [ *,  *,  *,  *,  *,  *,  *]
# [ *,  *,  *,  *,  *,  *,  *]
# [ 5, 14, 23, 32, 41, 50, 59],
# [ 4, 13, 22, 31, 40, 49, 58],
# [ 3, 12, 21, 30, 39, 48, 57],
# [ 2, 11, 20, 29, 38, 47, 56],
# [ 1, 10, 19, 28, 37, 46, 55],
# [ 0,  9, 18, 27, 36, 45, 54]

ntuple_std_list = [
    [0, 6, 7, 12, 13, 14, 19, 21],
    [13, 18, 19, 20, 21, 26, 27, 33],
    [1, 3, 4, 6, 7, 8, 9, 10],
    [7, 8, 9, 12, 15, 19, 25, 30],
    [4, 5, 9, 10, 11, 15, 16, 17],
    [1, 2, 3, 8, 9, 10, 16, 17],
    [3, 8, 9, 10, 11, 14, 15, 16],
    [0, 1, 2, 6, 8, 12, 13, 18],
    [25, 26, 27, 32, 33, 37, 38, 39],
    [3, 4, 8, 9, 11, 14, 15, 21],
    [2, 3, 4, 8, 9, 14, 15, 20],
    [18, 19, 24, 30, 31, 32, 36, 37],
    [3, 4, 8, 9, 10, 14, 15, 16],
    [5, 10, 11, 16, 17, 21, 22, 27],
    [4, 10, 15, 20, 21, 22, 27, 28],
    [18, 24, 25, 30, 31, 32, 37, 38],
    [11, 17, 21, 23, 27, 28, 33, 39],
    [21, 25, 26, 27, 32, 34, 35, 41],
    [22, 25, 26, 27, 30, 32, 33, 37],
    [4, 10, 11, 16, 20, 21, 22, 23],
    [0, 6, 7, 8, 12, 13, 14, 15],
    [17, 23, 28, 29, 32, 33, 34, 35],
    [0, 6, 7, 12, 18, 25, 32, 38],
    [2, 3, 4, 5, 8, 9, 10, 11],
    [27, 32, 33, 34, 37, 38, 39, 40],
    [4, 10, 16, 21, 26, 32, 33, 38],
    [0, 6, 7, 12, 13, 20, 27, 28],
    [0, 6, 12, 19, 25, 31, 32, 33],
    [1, 2, 6, 7, 13, 14, 15, 20],
    [1, 2, 5, 8, 11, 15, 16, 17],
    [13, 14, 16, 18, 21, 22, 23, 24],
    [2, 3, 9, 10, 11, 16, 17, 22],
    [15, 16, 17, 20, 22, 23, 25, 31],
    [15, 16, 17, 21, 22, 23, 28, 29],
    [24, 26, 30, 31, 32, 33, 36, 37],
    [12, 13, 18, 19, 20, 26, 27, 33],
    [1, 2, 3, 8, 9, 13, 14, 21],
    [13, 14, 18, 20, 24, 25, 31, 37],
    [14, 15, 16, 21, 26, 31, 38, 39],
    [1, 2, 6, 7, 12, 13, 14, 20],
    [4, 5, 10, 11, 17, 22, 23, 29],
    [2, 4, 5, 7, 9, 10, 14, 19],
    [5, 9, 10, 11, 15, 16, 21, 27],
    [1, 2, 3, 7, 8, 13, 14, 20],
    [1, 2, 8, 9, 14, 15, 21, 26],
    [22, 23, 29, 33, 34, 35, 38, 41],
    [13, 18, 19, 24, 25, 26, 31, 32],
    [27, 28, 29, 31, 32, 33, 37, 38],
    [10, 14, 15, 16, 17, 20, 21, 23],
    [4, 5, 9, 10, 15, 20, 21, 22],
    [13, 20, 25, 26, 27, 32, 34, 41],
    [30, 31, 33, 34, 36, 37, 38, 39],
    [11, 16, 23, 28, 34, 35, 40, 41],
    [3, 4, 10, 11, 14, 15, 16, 17],
    [15, 20, 21, 22, 26, 32, 33, 39],
    [18, 19, 25, 26, 31, 32, 34, 39],
    [4, 9, 11, 15, 16, 22, 23, 29],
    [26, 27, 31, 32, 33, 37, 38, 39],
    [20, 27, 28, 33, 34, 35, 40, 41],
    [1, 2, 7, 14, 20, 27, 28, 29],
    [8, 9, 10, 15, 16, 17, 22, 23],
    [9, 14, 15, 20, 21, 22, 27, 32],
    [1, 2, 3, 6, 7, 8, 9, 13],
    [10, 14, 15, 16, 20, 23, 25, 26],
    [0, 1, 2, 6, 7, 8, 13, 14],
    [1, 6, 7, 12, 13, 20, 26, 27],
    [8, 14, 20, 25, 26, 31, 33, 38],
    [20, 21, 26, 27, 28, 33, 35, 40],
    [2, 3, 4, 8, 9, 11, 16, 21],
    [1, 2, 3, 4, 5, 6, 11, 12],
]

# Convert to our bitboard convention
ntuple_std_list = [[x + 3 * (x // 6) for x in tup] for tup in ntuple_std_list]

In [ ]:
import torch
import torch.nn as nn

class TDConnect4AgentTorch:
    """TD n-tuple agent compatible with `Connect4Agent` Protocol.

    Implements:
      - score_all_moves(board) -> dict[int,int]
      - best_move(board) -> int
      - score_move(board, column, first_guess=0) -> int
    """

    def __init__(
        self,
        model_path: str | None=None,
        *,
        tie_break: str = "center",
    ) -> None:
        net2 = NTupleNetwork.load(model_path, device="cpu")
        net2.eval()
        self._tie_break = tie_break
        self._eval = net2

    # ---- Protocol method 1 ----
    def score_all_moves(self, board) -> dict[int, int]:
        """Return {col: score} for all legal moves. Illegal/full columns are excluded."""
        player_to_move = board.current_player() - 1  # BitBully: {1,2} -> {0,1}
        if player_to_move not in (0, 1):
            raise ValueError(f"Unexpected current_player(): {board.current_player()}")

        scores: dict[int, int] = {}

        for col in range(7):
            if not board.is_legal_move(col):
                continue

            score = self.score_move(board=board, column=col)
            scores[col] = score

        return scores

    # ---- Protocol method 2 ----
    def best_move(self, board) -> int:
        """Return best legal move using BitBully-like tie breaking."""
        scores = self.score_all_moves(board)
        if not scores:
            raise ValueError("No legal moves available.")

        best = max(scores.values())
        candidates = [c for c, v in scores.items() if v == best]

        if len(candidates) == 1:
            return candidates[0]

        if self._tie_break == "center":
            center = 3
            return min(candidates, key=lambda c: (abs(c - center), c))
        if self._tie_break == "left":
            return min(candidates)
        if self._tie_break == "right":
            return max(candidates)

        raise ValueError("tie_break must be one of: 'center', 'left', 'right'")

    # ---- Optional Protocol method ----
    def score_move(self, board, column: int, first_guess: int = 0) -> int:
        """Evaluate a single legal move. `first_guess` is accepted for Protocol compatibility."""
        _ = first_guess  # this TD agent ignores it
        if not board.is_legal_move(column):
            raise ValueError(f"Illegal move: column {column}")

        player_to_move = board.current_player() - 1
        after = board.play_on_copy(column)
        next_player = after.current_player() - 1

        if after.has_win():
            return 100
        else:
          all_tokens, active_tokens, moves_left = after._board.rawState()
          torch_board = BoardBatch(
              all_tokens=torch.tensor([all_tokens]),
              active_tokens=torch.tensor([active_tokens]),
              moves_left=torch.tensor([moves_left]),
          )

          score = float(self._eval.forward(torch_board)[0].item())

        if player_to_move == 1:
            score = -score

        return int(score * 100.0)

class NTupleNetwork(nn.Module):
    def __init__(self, n_tuple_list: list[list[int]]):
        super().__init__()

        self.M = len(n_tuple_list)
        self.N = len(n_tuple_list[0])
        self.K = 4**self.N

        # [M, N] bit indices
        self.n_tuple_tensor = torch.tensor(n_tuple_list, dtype=torch.int64)

        # Two players × M LUTs × K entries
        # 0 = Yellow, 1 = Red
        self.W = nn.Parameter(torch.zeros(2, self.M, self.K))

    def forward(self, board: "BoardBatch") -> torch.Tensor:
        """
        Returns:
            [B] tensor in [-1, 1]
        """
        # [B, M] table indices
        T = board.table_positions(self.n_tuple_tensor)
        T_mir = board.mirror().table_positions(self.n_tuple_tensor)
        B, M = T.shape
        dev = T.device

        # Active player per board: 0=Yellow, 1=Red
        # reuse your parity logic
        player_idx = ((board.moves_left.to(torch.int64) & 1) != 0).to(torch.int64)
        # shape: [B]

        # Pattern indices [M]
        m_idx = torch.arange(M, device=dev)

        # Gather:
        # W[player_idx[b], m, T[b,m]]
        w = self.W[
            player_idx.unsqueeze(1),  # [B,1]
            m_idx.unsqueeze(0),  # [1,M]
            T,  # [B,M]
        ]  # -> [B,M]
        w_mir = self.W[player_idx.unsqueeze(1), m_idx.unsqueeze(0), T_mir]

        # Sum over patterns (and symmetry): [B]
        score = (w + w_mir).sum(dim=1)
        return torch.tanh(score)

    @torch.no_grad()
    def save(self, path: str) -> None:
        """Save model weights + architecture-relevant metadata."""
        payload = {
            "state_dict": self.state_dict(),
            "n_tuple_list": self.n_tuple_tensor.cpu().tolist(),
        }
        torch.save(payload, path)

    @classmethod
    def load(
        cls,
        path: str,
        *,
        device: torch.device | str = "cpu",
        strict: bool = True,
    ) -> "NTupleNetwork":
        """
        Load model fully onto the specified device (CPU or GPU).
        """

        # 1) Always load checkpoint onto CPU first (safe & portable)
        payload = torch.load(path, map_location="cpu")

        if not isinstance(payload, dict) or "state_dict" not in payload:
            raise ValueError("Invalid checkpoint format.")

        n_tuple_list = payload.get("n_tuple_list")
        if n_tuple_list is None:
            raise ValueError("Checkpoint missing 'n_tuple_list'.")

        # 2) Construct model (still on CPU)
        model = cls(n_tuple_list=n_tuple_list)

        # 3) Load weights (still CPU tensors)
        model.load_state_dict(payload["state_dict"], strict=strict)

        # 4) Move EVERYTHING (parameters + buffers) in one go
        model = model.to(device)

        return model

import torch
import torch.nn.functional as F


def best_afterstate_values(
    board: "BoardBatch",
    net: "NTupleNetwork",
    *,
    moves_mask: torch.Tensor | None = None,
    randomize: torch.Tensor | None = None,  # [B] bool (epsilon-greedy)
    use_non_losing: bool = True,
    no_grad: bool = True,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Returns:
        chosen_mv:  [B] int64 one-hot move mask (landing square)
        chosen_val: [B] float32 value (reward if terminal else net score)
    """
    dev = board.all_tokens.device
    B = board.all_tokens.shape[0]

    # Which move set to iterate?
    if moves_mask is None:
        moves_mask = (
            board.generate_non_losing_moves()
            if use_non_losing
            else board.generate_legal_moves()
        )
    moves_mask = moves_mask.to(device=dev, dtype=torch.int64)

    yellow_to_move = (board.moves_left.to(torch.int64) & 1) == 0  # [B] bool

    neg_inf = torch.tensor(float("-inf"), device=dev)
    pos_inf = torch.tensor(float("inf"), device=dev)
    best_val = (
        torch
        .where(yellow_to_move, neg_inf, pos_inf)
        .to(torch.float32)
        .expand(B)
        .clone()
    )
    best_mv = torch.zeros((B,), device=dev, dtype=torch.int64)

    # Uniform random move via reservoir sampling over the iterated moves
    rand_mv = torch.zeros((B,), device=dev, dtype=torch.int64)
    rand_val = torch.full((B,), float("nan"), device=dev, dtype=torch.float32)
    seen = torch.zeros(
        (B,), device=dev, dtype=torch.int32
    )  # number of moves seen so far per board

    ctx = torch.no_grad() if no_grad else torch.enable_grad()
    with ctx:
        for mv in board.iter_move_masks(moves_mask):
            active = mv != 0
            if not active.any():
                break

            # Afterstate
            tmp = type(board)(
                all_tokens=board.all_tokens.clone(),
                active_tokens=board.active_tokens.clone(),
                moves_left=board.moves_left.clone(),
            )
            legal = tmp.play_masks(mv)
            active = active & legal  # defensive

            # Terminal reward: +1/-1/0 or NaN if not done
            r = tmp.reward().to(torch.float32)
            v = net(tmp).to(torch.float32)

            # tiebreak noise (optional)
            eps = 1e-4
            v = v + eps * torch.randn_like(v)

            val = torch.where(torch.isnan(r), v, r)  # [B]

            # --- greedy best update ---
            better = (
                torch.where(yellow_to_move, val > best_val, val < best_val) & active
            )
            best_val = torch.where(better, val, best_val)
            best_mv = torch.where(better, mv, best_mv)

            # --- reservoir sampling (uniform random among legal moves) ---
            # increment seen count for boards where this move exists
            seen = seen + active.to(seen.dtype)  # t in {1..}
            # replace current random choice with prob 1/seen
            # (uniform float u in [0,1); replace if u < 1/t)
            t = seen.to(torch.float32)
            replace = active & (torch.rand((B,), device=dev) < (1.0 / t))
            rand_mv = torch.where(replace, mv, rand_mv)
            rand_val = torch.where(replace, val, rand_val)

    if randomize is None:
        return best_mv, best_val

    randomize = randomize.to(device=dev, dtype=torch.bool)
    chosen_mv = torch.where(randomize, rand_mv, best_mv)
    chosen_val = torch.where(randomize, rand_val, best_val)
    return chosen_mv, chosen_val

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from bitbully import BitBully
from bitbully.agent_interface import Connect4Agent
import bitbully.bitbully_core as bbc
from bitbully import Board

bitbully_agent = BitBully(opening_book="12-ply-dist", tie_break="random")

In [ ]:
import logging
from techdays26.bitbully_arena import (
    BitBullyArena,
    ArenaConfig,
    AgentSpec,
    RandomAgent,
    Color,
    TimeControl,
    ArenaResult,
    GameRecord,
    AggregateRow,
    TerminationReason,
    SkippedPairing,
    MoveMeta,
    GamePlayers,
    GameConfig
)

def evaluate(td_agent, rnd_agent, bitbully_agent):
  # Logger is optional; arena is silent by default unless you configure logging.
  logger = logging.getLogger("bitbully.arena")
  logger.setLevel(logging.WARNING)  # warnings for illegal/exception/timeout


  cfg = ArenaConfig(
      agents=(
          AgentSpec(
              agent_id="random",
              agent=rnd_agent,
              colors=(Color.YELLOW, Color.RED),  # can play both
              epsilons=(0.00,),
          ),
          AgentSpec(
              agent_id="bitbully",
              agent=bitbully_agent,
              colors=(Color.YELLOW, Color.RED),  # can play both
              epsilons=(0.00, 0.1, 0.2, 0.3),
          ),
          AgentSpec(
              agent_id="ntuple",
              agent=td_agent,
              colors=(Color.YELLOW, Color.RED),  # can play both,
              epsilons=(0.00,),
          ),
      ),
      n_games=50,  # n games per pairing per ε per color-swap
      time_control=TimeControl(
          per_move_timeout_s=4.0,  # best-effort (measured after return)
          per_game_budget_s=45.0,  # seconds of total think time
      ),
      seed=None,
      log_scores=False,  # set True to also call score_all_moves() for logging
      use_tqdm=True,  # optional progress bar
      logger=logger,
  )

  # -----------------------------
  # Run arena
  # -----------------------------

  arena = BitBullyArena()
  result = arena.run(cfg)
  return result

from __future__ import annotations

from dataclasses import asdict
from typing import Any


def format_aggregate_table(result: Any) -> str:
    """
    Builds a nicely formatted table for `result.aggregates` and adds a final score:
      score = (+1 * yellow_wins) + (-1 * red_wins) + (0 * draws) = yellow_wins - red_wins

    Returns:
        A single string (ready to print or write to a file).
    """
    rows: list[dict[str, Any]] = []
    for r in result.aggregates:
        score = int(r.yellow_wins) - int(r.red_wins)

        rows.append({
            "yellow": r.agent_yellow,
            "red": r.agent_red,
            "eps_y": float(r.epsilon_yellow),
            "eps_r": float(r.epsilon_red),
            "games": int(r.games),
            "Y_w": int(r.yellow_wins),
            "R_w": int(r.red_wins),
            "D": int(r.draws),
            "score": score,
            "avg": (score / int(r.games)) if int(r.games) else 0.0,  # normalized [-1, 1]
            "timeouts": int(getattr(r, "timeouts", 0)),
            "illegal": int(getattr(r, "illegal_moves", 0)),
            "exc": int(getattr(r, "exceptions", 0)),
        })

    # Column order + formatting
    cols = [
        ("yellow",   "Y",       "{}"),
        ("red",      "R",       "{}"),
        ("eps_y",    "εY",      "{:.2f}"),
        ("eps_r",    "εR",      "{:.2f}"),
        ("games",    "G",       "{:d}"),
        ("Y_w",      "Ywin",    "{:d}"),
        ("R_w",      "Rwin",    "{:d}"),
        ("D",        "Draw",    "{:d}"),
        ("score",    "Score",   "{:d}"),
        ("avg",      "Avg",     "{:+.3f}"),
        ("timeouts", "TO",      "{:d}"),
        ("illegal",  "Ill",     "{:d}"),
        ("exc",      "Exc",     "{:d}"),
    ]

    # Pre-render cells to compute widths
    rendered: list[list[str]] = []
    for row in rows:
        rendered.append([fmt.format(row[key]) for key, _, fmt in cols])

    widths = []
    for j, (_, header, _) in enumerate(cols):
        w = len(header)
        for i in range(len(rendered)):
            w = max(w, len(rendered[i][j]))
        widths.append(w)

    def pack_line(parts: list[str]) -> str:
        return " | ".join(p.ljust(widths[i]) for i, p in enumerate(parts))

    header = pack_line([h for _, h, _ in cols])
    sep = "-+-".join("-" * w for w in widths)

    lines = [header, sep]
    for cells in rendered:
        lines.append(pack_line(cells))

    return "\n".join(lines)

legend = (
    "Table legend:\n"
    "Y      = Yellow agent ID\n"
    "R      = Red agent ID\n"
    "εY     = Epsilon value used for the Yellow agent\n"
    "εR     = Epsilon value used for the Red agent\n"
    "G      = Number of games played for this pairing/configuration\n"
    "Ywin   = Number of games won by Yellow\n"
    "Rwin   = Number of games won by Red\n"
    "Draw   = Number of draws\n"
    "Score  = Final score = Ywin − Rwin  (Yellow win = +1, Red win = −1, Draw = 0)\n"
    "Avg    = Normalized score per game = Score / G  (range [-1, +1])\n"
    "TO     = Games lost due to timeout\n"
    "Ill    = Games lost due to illegal moves\n"
    "Exc    = Games lost due to exceptions\n"
)

# print(legend)

# Usage:
#table_str = format_aggregate_table(result)
#print(table_str)
# or write it to a file:
# with open("arena_summary.txt", "a", encoding="utf-8") as f:
#     f.write(table_str + "\n")

from __future__ import annotations

import json
from dataclasses import asdict, is_dataclass
from enum import Enum
from pathlib import Path
from typing import Any


# -----------------------------
# JSON helpers
# -----------------------------

def _to_jsonable(obj: Any) -> Any:
    """Convert dataclasses/enums/tuples into JSON-serializable structures."""
    if is_dataclass(obj):
        return {k: _to_jsonable(v) for k, v in asdict(obj).items()}
    if isinstance(obj, Enum):
        # store enum values (Color is int, TerminationReason is str)
        return obj.value
    if isinstance(obj, tuple):
        return [_to_jsonable(x) for x in obj]
    if isinstance(obj, list):
        return [_to_jsonable(x) for x in obj]
    if isinstance(obj, dict):
        return {str(k): _to_jsonable(v) for k, v in obj.items()}
    return obj


def save_arena_result(result: ArenaResult, path: str | Path) -> None:
    path = Path(path)
    payload = {
        "version": 1,
        "result": _to_jsonable(result),
    }
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


# -----------------------------
# Restore dataclasses
# -----------------------------

def load_arena_result(path: str | Path) -> ArenaResult:
    path = Path(path)
    payload = json.loads(path.read_text(encoding="utf-8"))

    if payload.get("version") != 1:
        raise ValueError(f"Unsupported arena result version: {payload.get('version')}")

    d = payload["result"]

    def mk_game_players(x: dict[str, Any]) -> GamePlayers:
        return GamePlayers(yellow_id=x["yellow_id"], red_id=x["red_id"])

    def mk_game_config(x: dict[str, Any]) -> GameConfig:
        return GameConfig(
            players=mk_game_players(x["players"]),
            epsilon_by_agent={k: float(v) for k, v in x["epsilon_by_agent"].items()},
            seed=int(x["seed"]),
        )

    def mk_move_meta(x: dict[str, Any]) -> MoveMeta:
        return MoveMeta(
            ply=int(x["ply"]),
            player=Color(int(x["player"])),
            agent_id=str(x["agent_id"]),
            epsilon=float(x["epsilon"]),
            move=int(x["move"]),
            was_epsilon_random=bool(x["was_epsilon_random"]),
            elapsed_s=float(x["elapsed_s"]),
            remaining_budget_s=(None if x["remaining_budget_s"] is None else float(x["remaining_budget_s"])),
            scores=(None if x.get("scores") is None else {int(k): int(v) for k, v in x["scores"].items()}),
        )

    def mk_game_record(x: dict[str, Any]) -> GameRecord:
        return GameRecord(
            game_cfg=mk_game_config(x["game_cfg"]),
            moves=tuple(int(m) for m in x["moves"]),
            move_meta=tuple(mk_move_meta(mm) for mm in x["move_meta"]),
            winner=(None if x["winner"] is None else Color(int(x["winner"]))),
            termination=TerminationReason(str(x["termination"])),
            detail=(None if x.get("detail") is None else str(x["detail"])),
        )

    def mk_skipped_pairing(x: dict[str, Any]) -> SkippedPairing:
        # reason defaults to INCOMPATIBLE_CONSTRAINTS in your dataclass, but store/load it anyway
        return SkippedPairing(
            agent_a=str(x["agent_a"]),
            agent_b=str(x["agent_b"]),
            reason=TerminationReason(str(x.get("reason", TerminationReason.INCOMPATIBLE_CONSTRAINTS.value))),
            detail=(None if x.get("detail") is None else str(x["detail"])),
        )

    def mk_aggregate_row(x: dict[str, Any]) -> AggregateRow:
        return AggregateRow(
            agent_yellow=str(x["agent_yellow"]),
            agent_red=str(x["agent_red"]),
            epsilon_yellow=float(x["epsilon_yellow"]),
            epsilon_red=float(x["epsilon_red"]),
            games=int(x["games"]),
            yellow_wins=int(x["yellow_wins"]),
            red_wins=int(x["red_wins"]),
            draws=int(x["draws"]),
            timeouts=int(x["timeouts"]),
            illegal_moves=int(x["illegal_moves"]),
            exceptions=int(x["exceptions"]),
        )

    return ArenaResult(
        games=tuple(mk_game_record(gr) for gr in d["games"]),
        aggregates=tuple(mk_aggregate_row(ar) for ar in d["aggregates"]),
        skipped=tuple(mk_skipped_pairing(sp) for sp in d["skipped"]),
    )


# -----------------------------
# Usage
# -----------------------------
# arena = BitBullyArena()
# result = arena.run(cfg)
# save_arena_result(result, "arena_result.json")
# result2 = load_arena_result("arena_result.json")


In [ ]:
from pathlib import Path

pre_trained_model: str | None = None
train_folder: str | Path = "/content/drive/MyDrive/models/exp_20260207/"
n_evaluate = 1000

train_folder = Path(train_folder)
train_folder.mkdir(parents=True, exist_ok=True)

B = 30000
epsilon = 0.1

# load pre-trained, if desired:
if pre_trained_model:
    ntuple_net = NTupleNetwork.load(
        pre_trained_model,
        device=device,
    )
    # ntuple_net.eval() # DO NOT DO DURING TRAINING
else:
    ntuple_net = NTupleNetwork(n_tuple_list=ntuple_std_list).to(device)

torch_board = BoardBatch.empty(B, device)
bb_board = [bbc.BoardCore() for _ in range(B)]

assert ntuple_net.training, "Model should be in training mode!"

# opt = torch.optim.Adam(
#    ntuple_net.parameters(),
#    lr=1e-3,        # start here
#    betas=(0.9, 0.999),
#    eps=1e-8,
# )

# opt = torch.optim.AdamW(
#    ntuple_net.parameters(),
#    lr=1e-5,
#    weight_decay=0.01,  # optional, small
# )

opt = torch.optim.Adam(ntuple_net.parameters(), lr=1e-2) # used to be 1e-3
scheduler = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.9999)

all_results = []
losses = []
for step in range(100000):
    V_old = ntuple_net(torch_board)
    randomize = (
        torch.rand((B,), device=torch_board.all_tokens.device) < epsilon
    )  # [B] bool

    with torch.no_grad():
        best_mv, V_new = best_afterstate_values(
            torch_board,
            ntuple_net,
            moves_mask=None,
            randomize=randomize,
            use_non_losing=False,
            no_grad=True,
        )

    # At any time, any move has to be legal:
    legal = torch_board.play_masks(best_mv)
    if False:
        assert all(legal == True) and not any(legal == False)

    # If the games are over, there has to be a reward
    if False:
        assert (torch_board.done() & torch.isnan(torch_board.reward())).sum() == 0
    done = torch_board.done()

    # Update only for afterstates which are terminal states or not random moves
    update_mask = done | (~randomize)

    loss = torch.nn.functional.mse_loss(V_old[update_mask], V_new[update_mask])

    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    scheduler.step()

    losses.append(loss.item())
    if step % 100 == 0:
        lr = opt.param_groups[0]["lr"]
        print(f"step {step:6d} | lr = {lr:.3e} | loss = {loss.item():.6f}")

    if step % n_evaluate == 0:
        print("evaluate...")
        # Save the weights to a file
        weights_path = f"{train_folder}/step_{step}_model_weights_loss_{loss.item():.3f}.pt"
        ntuple_net.save(weights_path)
        td_agent = TDConnect4AgentTorch( # TODO: Allow to pass object (or copy) directly
            model_path=weights_path
        )
        result = evaluate(td_agent, RandomAgent(), bitbully_agent)
        save_arena_result(result, f"{train_folder}/step_{step}_arena_result.json") # TODO: Integrate into class

        out_file = Path(train_folder / "0_log.txt")

        with out_file.open("a", encoding="utf-8") as f:
            if step == 0:
                f.write(legend + "\n\n")
            lr = opt.param_groups[0]["lr"]
            f.write("=====================================================" * 2 + "\n")
            f.write(f"step {step:6d} | lr = {lr:.3e} | loss = {loss.item():.5f}\n\n")
            f.write(format_aggregate_table(result))
            f.write("=====================================================" * 2 + "\n\n")


    # Just some verification
    if False:
        for b_idx in range(B):
            mv = best_mv[b_idx].item()
            a = move_mask_to_column(mv)
            assert bb_board[b_idx].play(a)
            bb_done = bb_board[b_idx].hasWin() or bb_board[b_idx].movesLeft() <= 0

            assert bb_done == done[b_idx].item(), f"Problem with {b_idx}"
            if bb_done:
                if bb_board[b_idx].hasWin() and bb_board[b_idx].movesLeft() % 2 == 0:
                    # print(best_val[b_idx].item(), bb_board[b_idx])
                    assert V_new[b_idx].item() == -1
                elif bb_board[b_idx].hasWin() and bb_board[b_idx].movesLeft() % 2 == 1:
                    assert V_new[b_idx].item() == 1
                    # print(best_val[b_idx].item(), bb_board[b_idx])
                elif bb_board[b_idx].movesLeft() <= 0:
                    assert V_new[b_idx].item() == 0
                bb_board[b_idx].setRawState(0, 0, 42)

    if done.any():
        torch_board.reset(done)

step      0 | lr = 9.999e-03 | loss = 0.000000
evaluate...


BitBullyArena: 100%|██████████| 900/900 [00:37<00:00, 24.06game/s]


step    100 | lr = 9.900e-03 | loss = 0.191849
step    200 | lr = 9.801e-03 | loss = 0.087358
step    300 | lr = 9.703e-03 | loss = 0.109112
step    400 | lr = 9.607e-03 | loss = 0.133101
step    500 | lr = 9.511e-03 | loss = 0.225295
step    600 | lr = 9.417e-03 | loss = 0.095951
step    700 | lr = 9.323e-03 | loss = 0.118558
step    800 | lr = 9.230e-03 | loss = 0.242427
step    900 | lr = 9.138e-03 | loss = 0.141312
step   1000 | lr = 9.047e-03 | loss = 0.226287
evaluate...


BitBullyArena: 100%|██████████| 900/900 [00:53<00:00, 16.68game/s]


step   1100 | lr = 8.957e-03 | loss = 0.204385
step   1200 | lr = 8.868e-03 | loss = 0.287604
step   1300 | lr = 8.780e-03 | loss = 0.304410
step   1400 | lr = 8.693e-03 | loss = 0.150778
step   1500 | lr = 8.606e-03 | loss = 0.099212
step   1600 | lr = 8.521e-03 | loss = 0.086219
step   1700 | lr = 8.436e-03 | loss = 0.089833
step   1800 | lr = 8.352e-03 | loss = 0.085637
step   1900 | lr = 8.269e-03 | loss = 0.104211
step   2000 | lr = 8.186e-03 | loss = 0.096712
evaluate...


...

BitBullyArena: 100%|██████████| 900/900 [01:31<00:00,  9.80game/s]


step  27100 | lr = 6.652e-04 | loss = 0.038221
step  27200 | lr = 6.586e-04 | loss = 0.044995
step  27300 | lr = 6.520e-04 | loss = 0.038560
step  27400 | lr = 6.456e-04 | loss = 0.038890
step  27500 | lr = 6.391e-04 | loss = 0.039715
step  27600 | lr = 6.328e-04 | loss = 0.035542
step  27700 | lr = 6.265e-04 | loss = 0.035261
step  27800 | lr = 6.202e-04 | loss = 0.035912
step  27900 | lr = 6.141e-04 | loss = 0.035683
step  28000 | lr = 6.080e-04 | loss = 0.031342
evaluate...


BitBullyArena: 100%|██████████| 900/900 [01:32<00:00,  9.73game/s]


step  28100 | lr = 6.019e-04 | loss = 0.033099
step  28200 | lr = 5.959e-04 | loss = 0.034187
step  28300 | lr = 5.900e-04 | loss = 0.033100
step  28400 | lr = 5.841e-04 | loss = 0.031669
step  28500 | lr = 5.783e-04 | loss = 0.032045
step  28600 | lr = 5.725e-04 | loss = 0.036276
step  28700 | lr = 5.669e-04 | loss = 0.033799
step  28800 | lr = 5.612e-04 | loss = 0.036096
step  28900 | lr = 5.556e-04 | loss = 0.038173
step  29000 | lr = 5.501e-04 | loss = 0.035788
evaluate...


BitBullyArena:  43%|████▎     | 389/900 [00:17<00:43, 11.63game/s]

In [ ]:
result = load_arena_result(f"{train_folder}/arena_result.json")

In [ ]:
if False: # Old:
  print("==================================================="*4)
  print("Skipped pairings:", len(result.skipped))
  print("Games played:", len(result.games))
  print("Aggregate rows:", len(result.aggregates))

  # Print aggregate rows (one row per (yellow_agent, red_agent, eps_y, eps_r))
  for row in result.aggregates:
      print(row)

  # Look at a single game record (includes move list + per-move metadata)
  g0 = result.games[0]
  print("First game:", g0.game_cfg)
  print("Termination:", g0.termination, "winner:", g0.winner)
  print("Moves:", g0.moves)
  print("First move meta:", g0.move_meta[0])
  print("==================================================="*4, "\n")

In [ ]:
agents: dict[str, Connect4Agent] = {
    "BitBully-12ply": bitbully_agent,
    "TD-Agent": td_agent,
}

In [ ]:
from bitbully.gui_c4 import GuiC4

%matplotlib ipympl

c4gui = GuiC4(agents=agents, autoplay=True)

# Display everything
display(c4gui.get_widget())